In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys; sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from prediction.config import FEATURE_SOURCES
from prediction import sources, features
from core.config import INTERIM_DIR, UCR_YEAR

In [2]:
# What sources are declared, and how each is fetched
for name, src in FEATURE_SOURCES.items():
    print(f"{name:12} backend={src.backend:4} location={src.location:10} "
          f"key={src.key_col:6} cols={src.feature_cols}")

vacancy      backend=bq   location=vacancy    key=geoid  cols=('vacant_pct',)
liens        backend=bq   location=liens      key=geoid  cols=('clip_liens_pct',)


In [ ]:
# Run only when the BQ staging tables don't exist yet or upstream data changed.
# This executes the CREATE OR REPLACE DDL — skip if already built. Left commented on purpose.
#for name in ["vacancy", "liens"]:
#    sources.run_bq_build(name)
#    print(f"built {name}")

In [3]:
# Uncached single pull to confirm BQ auth + schema before running the full pipeline
liens = sources.run_bq_pull("liens")
print("liens pull:", liens.shape)
liens.head()

/home/eprashar_solutions_corelogic_com/.cache/pypoetry/virtualenvs/crime-idx-2026-v3aYThD0-py3.12/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


liens pull: (241456, 4)


,geoid,clip_liens_pct,total_clips,clip_w_liens
0,120830017001,0.09,1090,1
1,390130107003,0.09,1134,1
2,080410033052,0.09,1066,1
3,291833122042,0.09,1110,1
4,220050301041,0.09,1071,1


In [ ]:
# Load liens and vacancy parquet files
for name, src in FEATURE_SOURCES.items():
    df = sources.pull_source(src, refresh=False)
    print(f"{name:12} {df.shape}  cols={list(df.columns)}")

/home/eprashar_solutions_corelogic_com/.cache/pypoetry/virtualenvs/crime-idx-2026-v3aYThD0-py3.12/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


vacancy      (241456, 4)  cols=['geoid', 'vacant_pct', 'vacant_addr', 'total_addr']


/home/eprashar_solutions_corelogic_com/.cache/pypoetry/virtualenvs/crime-idx-2026-v3aYThD0-py3.12/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


liens        (241456, 4)  cols=['geoid', 'clip_liens_pct', 'total_clips', 'clip_w_liens']


In [5]:
# Load demographic features
demo = features.build_demographic_features(refresh=False)
print("demographic:", demo.shape)
demo.head()

demographic: (242335, 8)


,geoid,det_pct,in_household_pct,moved1yr_pct,Division,city_centers_dist,pop_est_5mile,pop_ch_1mile
0,020130001001,73.279352,65.293602,16.042465,9.0,50.0,2066.0,-37.826087
1,020130001002,57.459677,65.293602,16.042465,9.0,50.0,884.0,-11.588785
2,020130001003,75.638051,65.293602,16.042465,9.0,50.0,674.0,-32.747604
3,020160001001,33.962264,66.310160,19.562244,9.0,50.0,1023.0,-6.250000
4,020160002001,9.478673,68.261851,24.745302,9.0,50.0,4411.0,-0.500835


In [9]:
# Assemble features
feats = features.assemble_features(refresh=False)
print("bg_predictors:", feats.shape)
feats.head(20)

bg_predictors: (242335, 10)


,geoid,det_pct,in_household_pct,moved1yr_pct,Division,city_centers_dist,pop_est_5mile,pop_ch_1mile,vacant_pct,clip_liens_pct
0,020130001001,73.279352,65.293602,16.042465,9.0,50.000000,2066.0,-37.826087,NaN,NaN
1,020130001002,57.459677,65.293602,16.042465,9.0,50.000000,884.0,-11.588785,NaN,NaN
2,020130001003,75.638051,65.293602,16.042465,9.0,50.000000,674.0,-32.747604,NaN,NaN
3,020160001001,33.962264,66.310160,19.562244,9.0,50.000000,1023.0,-6.250000,NaN,NaN
4,020160002001,9.478673,68.261851,24.745302,9.0,50.000000,4411.0,-0.500835,0.000000,0.00
5,020160002002,38.165138,68.261851,24.745302,9.0,50.000000,4411.0,10.603589,0.000000,0.00
6,020200001011,75.589354,98.301445,16.106719,9.0,30.000000,9161.0,3.715883,0.000000,0.78
7,020200001012,87.484812,98.301445,16.106719,9.0,50.000000,9161.0,5.971610,0.000000,1.45
8,020200001021,72.164948,99.100592,9.289486,9.0,30.000000,10816.0,3.649635,0.000000,0.90
9,020200001022,34.243697,99.100592,9.289486,9.0,50.000000,28512.0,-4.620750,0.000000,2.55


In [7]:
# geoid should be a 12-char string, unique per row
print("geoid dtype :", feats["geoid"].dtype)
print("geoid lengths:", feats["geoid"].str.len().value_counts().to_dict())
print("duplicate geoids:", feats["geoid"].duplicated().sum())

# Non-null coverage per column — low coverage on vacancy/liens flags join or fill issues
print("\nNon-null coverage:")
print((feats.notna().mean() * 100).round(1).astype(str) + "%")

geoid dtype : str
geoid lengths: {12: 242335}
duplicate geoids: 0

Non-null coverage:
geoid                100.0%
det_pct              100.0%
in_household_pct     100.0%
moved1yr_pct         100.0%
Division             100.0%
city_centers_dist     98.9%
pop_est_5mile         98.8%
pop_ch_1mile          98.8%
vacant_pct            99.1%
clip_liens_pct        99.6%
dtype: str


In [8]:
print("Cached source pulls:")
for p in sorted((INTERIM_DIR / "sources").glob("*.parquet")):
    print(" ", p.relative_to(INTERIM_DIR.parent), f"{p.stat().st_size/1e6:.1f} MB")

feat_path = INTERIM_DIR / "features" / "bg_predictors.parquet"
print("\nFeature matrix written:", feat_path.exists(), "→", feat_path)

Cached source pulls:
  interim/sources/demographic.parquet 8.0 MB
  interim/sources/liens.parquet 2.6 MB
  interim/sources/vacancy.parquet 3.3 MB

Feature matrix written: True → /home/eprashar_solutions_corelogic_com/crime-idx-2026/data/interim/features/bg_predictors.parquet
